# Heritage Short: Wan 2.2 TI2V-5B image-to-video on Kaggle T4 x2 (both GPUs in parallel)
Pushed by `pipeline/shots.py --parallel` via `pipeline/kaggle_job.py`. The JOBS cell is filled in at push time.
The keyframes and `wan_worker.py` arrive as the private dataset `<user>/heritage-short-wan-in` (checked by sha256).
Each GPU runs its **own full pipeline** on its share of the shots (no model is split across GPUs, FAILURES.md #22),
with the same settings as the proven single-GPU notebook `wan_i2v.ipynb`:
1. phase 1, alone: all prompts through the text encoder, saved, and the encoder process exits (frees 11 GB);
2. phase 2: one worker per GPU, started one after the other (only one pipeline loads at a time: CPU RAM);
   lanes follow the priority order, e.g. 9,5,4,3 -> GPU 0 renders 9 then 4, GPU 1 renders 5 then 3;
3. if a worker dies (RAM, device error), the shots it did not finish run one at a time on one GPU, in order.
Each clip is written to `out/<shot>.mp4` the moment it is done; `timing.jsonl` gets one line per attempt.

In [ ]:
# @JOBS (replaced at push time)
JOBS = {"job_id": "local-test", "config": {}, "jobs": []}

In [ ]:
import glob, hashlib, json, os, subprocess, sys, time
T0 = time.time()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # inherited by the workers
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/out", exist_ok=True)
_log = open(f"{OUT}/timing.jsonl", "a")
def log(**kw):
    """One JSON line per event; the workers append their attempts to the same file."""
    _log.write(json.dumps({"job_id": JOBS["job_id"], "worker": "main", **kw}) + "\n"); _log.flush(); print(kw, flush=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.39.0", "ftfy", "imageio-ffmpeg"], check=True)
cfg = JOBS["config"]
WORKER = next((p for p in glob.glob("/kaggle/input/**/wan_worker.py", recursive=True)
               if hashlib.sha256(open(p, "rb").read()).hexdigest() == cfg["worker_sha256"]), None)
if WORKER is None:
    log(id=None, event="error", error="wan_worker.py with the expected sha256 not found (stale dataset version?)")
    raise SystemExit("worker missing")
JOBS_PATH, EMB = "/tmp/wan_jobs.json", "/tmp/wan_emb.pt"
json.dump(JOBS, open(JOBS_PATH, "w"))
GPUS = [g.strip() for g in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                          capture_output=True, text=True).stdout.splitlines() if g.strip()]
log(id=None, event="setup", seconds=round(time.time() - T0, 1), gpu=", ".join(GPUS), gpus=len(GPUS))

In [ ]:
# phase 1: text encoder alone, in its own process (its memory is returned when it exits)
r = subprocess.run([sys.executable, WORKER, "encode", JOBS_PATH, EMB])
if r.returncode != 0 or not os.path.exists(EMB):
    log(id=None, event="error", error=f"text encoding failed (exit {r.returncode})")
    raise SystemExit("text encoding failed")

In [ ]:
# phase 2: one full pipeline per GPU, then a one-GPU pass for anything a crashed worker left
def done_shots():
    lines = [json.loads(x) for x in open(f"{OUT}/timing.jsonl") if x.strip()]
    return {ln["shot"] for ln in lines if ln.get("event") == "job_done" and ln.get("job_id") == JOBS["job_id"]}

def start(gpu, shots):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen([sys.executable, WORKER, "render", JOBS_PATH, EMB, str(gpu), ",".join(shots)], env=env)

order = cfg["order"]
lanes = [order[0::2], order[1::2]] if len(GPUS) >= 2 and len(order) > 1 else [order]
log(id=None, event="plan", lanes=lanes)
procs = []
for gpu, shots in enumerate(lanes):
    p = start(gpu, shots)
    procs.append((gpu, p))
    t = time.time()
    while gpu + 1 < len(lanes) and p.poll() is None and not os.path.exists(f"/tmp/wan_loaded_{gpu}") \
            and time.time() - t < 900:
        time.sleep(5)  # load one pipeline at a time (CPU RAM)
for gpu, p in procs:
    log(id=None, event="worker_exit", gpu_index=gpu, code=p.wait())

left = [s for s in order if s not in done_shots()]
if left:
    log(id=None, event="sequential_fallback", shots=left)
    p = start(0, left)
    log(id=None, event="worker_exit", gpu_index=0, code=p.wait(), sequential=True)
log(id=None, event="done", seconds=round(time.time() - T0, 1), shots_done=sorted(done_shots()))